# Pipeline 2: YOLOv8-Nano + MiDaS Small (LiDAR Scale-Shift Aligned)
**Sorumlu:** Kişi 2 (Emirhan) | **Branch:** `feat/pipeline2-yolov8-midas`

Bu notebook **yalnızca Pipeline 2** için gereken adımları çalıştırır:
1. GitHub'dan kendi branch'ini çeker ve ortak Google Drive klasörüne bağlanır.
2. Resmi KITTI verisini (LiDAR dahil) indirir ve `01_prepare_data.py` çalıştırır (repodaki sabit `data/splits/*.txt` korunur).
3. `YOLOv8-Nano` modelini 3 random seed (`42, 123, 999`) ile eğitir.
4. Eğitilen ağırlıkları (`checkpoints/yolov8n_kitti_seed*.pt`) ortak Google Drive'a kaydeder.
5. MiDaS Small + per-image LiDAR scale-shift hizalama doğruluğunu test eder.

In [ ]:
!nvidia-smi
from google.colab import drive
import os

drive.mount('/content/drive')
SHARED_DRIVE = '/content/drive/MyDrive/CS423_Shared_Project'
os.makedirs(f'{SHARED_DRIVE}/checkpoints', exist_ok=True)
os.makedirs(f'{SHARED_DRIVE}/results', exist_ok=True)
print('Ortak Drive klasörü hazır:', SHARED_DRIVE)

In [ ]:
REPO_URL = "https://github.com/nehirvardar/cs423-assistive-navigation.git"
BRANCH = "pipeline-2"  # main'den açtığınız branch adı

!rm -rf /content/project
!git clone -b {BRANCH} {REPO_URL} /content/project
%cd /content/project
!pip install -q -r requirements.txt
!apt-get update -qq && apt-get install -qq -y aria2

In [ ]:
# MiDaS hizalaması LiDAR derinliği gerektirdiği için tam KITTI indirmesi yapılır
!python scripts/download_kitti.py
!python scripts/01_prepare_data.py

In [ ]:
# YOLOv8-Nano'yu 3 random seed ile eğit
!python scripts/03_train_yolo.py --model yolov8n --seeds 42 123 999 --epochs 50 --batch 16 --device 0

# Ağırlıkları ortak Google Drive'a yedekle
!cp -v checkpoints/yolov8n_kitti_seed*.pt {SHARED_DRIVE}/checkpoints/

In [ ]:
# Pipeline 2 hızlı doğrulama (İlk 50 test karesinde MiDaS LiDAR alignment testi)
!python scripts/04_evaluate_all.py --split test --max-frames 50 --device cuda
!cp -r results {SHARED_DRIVE}/